# 3. Revisión de anotaciones y criterios

Revisión reproducible de las 17 cajas de clases 1/2, las 37 cajas pequeñas positivas,
la caja degenerada y una muestra de 18 imágenes con etiqueta vacía.
No cambia datos ni split. Completar `INTEGRANTE` y ejecutar todo.

Los [criterios de trabajo](../docs/criterios_datos.md) y el
[registro de revisión](../docs/revision_anotaciones.csv) están versionados.
Las observaciones son de una revisión visual de Codex; N06 y N16 requieren segunda revisión.


In [ ]:
# --- Setup: correr siempre primero ---
import os
import subprocess
import sys
from pathlib import Path

INTEGRANTE = os.environ.get("PDI_INTEGRANTE", "nombre")  # reemplazar por tu nombre

if "google.colab" in sys.modules:
    repo = Path("/content/PDI-TP")
    if repo.is_dir():
        subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/candeperles/PDI-TP.git", str(repo)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")], check=True)
else:
    # Funciona al abrir Jupyter desde la raíz o desde notebooks/.
    repo = next((p for p in (Path.cwd(), Path.cwd().parent)
                 if (p / "pdi_tp" / "entorno.py").is_file()), None)
    if repo is None:
        raise FileNotFoundError("Abrí Jupyter desde la raíz del repositorio o desde notebooks/.")
os.chdir(repo)
sys.path.insert(0, str(repo))

from pdi_tp import entorno, datos, resultados, split
rutas = entorno.preparar(INTEGRANTE)


## Construir los mismos casos

Se incluyen todas las cajas minoritarias, pequeñas positivas e inválidas. Los negativos
se muestrean con semilla 42, tres por combinación fuente/split disponible. Es una
muestra exploratoria, no una estimación de error sobre todos los negativos.


In [ ]:
from pdi_tp import validacion, revision
import pandas as pd
from IPython.display import Image as MostrarImagen

informe = validacion.validar(rutas)
casos = revision.seleccionar_casos(informe, semilla=42, negativos_por_estrato=3)
registro = pd.read_csv(rutas.repo / "docs" / "revision_anotaciones.csv", dtype={"linea": "Int64", "clase": "Int64"})
columnas = list(casos.columns)
pd.testing.assert_frame_equal(casos.reset_index(drop=True), registro[columnas].reset_index(drop=True),
                              check_dtype=False, check_exact=False, atol=1e-6, rtol=0)
revision_completa = casos.merge(registro[["caso", "observacion", "decision", "estado"]],
                               on="caso", validate="one_to_one")
display(casos.groupby(["categoria", "split"]).size().rename("casos").to_frame())
salida = rutas.analisis / "revision_anotaciones"
galerias = revision.exportar_galerias(casos, informe, rutas, salida)
revision_completa.to_csv(salida / "revision_completa.csv", index=False, encoding="utf-8")
print(f"Galerías y registro: {salida}")


## Clases 1 y 2: confianza de origen

El [artículo original, Technical Validation](https://www.nature.com/articles/s41597-023-02539-8)
define 0/1/2 como confianza alta/moderada/baja en `solarpanels_hd`.
El [cotejo con Figshare](../docs/cotejo_clases.csv) verificó las 17 cajas locales.
No son tipos distintos de panel. En otras fuentes no se infiere confianza a partir del ID 0.

Cada caso muestra la imagen completa y un zoom con contexto. El zoom no agrega resolución.
El criterio es una clase `panel` en un futuro derivado, conservando confianza y clase originales.


In [ ]:
display(pd.read_csv(rutas.repo / "docs" / "cotejo_clases.csv"))
display(revision_completa[revision_completa["categoria"] == "clase_minoritaria"])
for archivo in galerias:
    if archivo.name.startswith("clase_minoritaria_"):
        display(MostrarImagen(filename=str(archivo)))


## Recortes pequeños y caja nula

Se conservan las 37 cajas de tamaño positivo, sin filtro automático a 3 px. Todas
tocan el borde dentro del redondeo de coordenadas. Quitar todas dejaría cuatro
imágenes positivas sin cajas. Z01 tiene ancho cero: solo esa caja se excluirá
cuando se prepare el derivado, manteniendo la imagen y las siete cajas restantes.


In [ ]:
display(revision_completa[revision_completa["categoria"].isin(["caja_pequena", "caja_invalida"])])
for archivo in galerias:
    if archivo.name.startswith(("caja_pequena_", "caja_invalida_")):
        display(MostrarImagen(filename=str(archivo)))


## Muestra con etiqueta vacía

Dieciocho imágenes cubren `geo`, `kingston` y `solarpanels_hd` en ambos splits.
PV01 no aporta negativos. N06 y N16 quedan como ambiguos: revisar con contexto
adicional antes de fijar el conjunto de evaluación, sin reclasificar por intuición.


In [ ]:
display(revision_completa[revision_completa["categoria"] == "muestra_sin_paneles"])
for archivo in galerias:
    if archivo.name.startswith("muestra_sin_paneles_"):
        display(MostrarImagen(filename=str(archivo)))


## Estado de las decisiones

Los criterios están en `docs/criterios_datos.md`. No se aplicó aún la fusión de clases
ni la exclusión de la caja nula: son reglas para una futura preparación reproducible.
No se cambiaron originales, negativos ni el split. Registrar la segunda revisión de
N06/N16 y la confirmación de alcance de la cátedra antes de cerrar la referencia.
